# Notebook 4 — Dataset Assembly
### Multimodal Driving Risk Prediction | MSc Business Analytics | Dublin Business School

---

## What This Notebook Does
Integrates all feature groups from Notebooks 1, 2, and 3 into one unified multimodal dataset.
Three additional composite features are engineered here.
A mandatory quality check confirms no target leakage before the dataset is saved.

## Key Steps
1. **Load** prepared metadata (NB1), ego-motion features (NB2), visual features (NB3)
2. **Merge** all tables on `clip_id` using inner joins
3. **Engineer** three composite features
4. **Encode** `lighting_condition` from brightness bins
5. **Check** that `anomaly_score` is absent — mandatory leakage prevention gate
6. **Validate** — confirm shape, zero missing values, correct column count
7. **Save** final dataset to Google Drive

## Engineered Features
| Feature | Formula | Purpose |
|---|---|---|
| `aggressive_score` | Sum of StandardScaler-normalised max_speed + max_acceleration + hard_braking_events + max_jerk | Composite driving aggression indicator — scaled to prevent dominance |
| `env_complexity` | Weighted sum of detected object counts | Scene complexity measure |
| `object_density` | `total_objects / (env_complexity + 1)` | Normalised object density per scene |

## Critical Quality Gate
Step 5 runs a **programmatic check** confirming `anomaly_score` is absent from the dataset.
If it appears, the pipeline halts — this prevents Notebook 1 sampling scores leaking into Notebook 5 labels.

## Input
- `prepared_dataset_2000.csv` — from Notebook 1
- `ego_features_2000.csv` — from Notebook 2
- `visual_features_final.csv` — from Notebook 3

## Output
- `final_dataset.csv` — **1,899 clips × 48 features** — the complete multimodal dataset

## Feature Groups in Final Dataset
| Group | Features | Count |
|---|---|---|
| Ego / Behaviour | speed, acceleration, jerk, braking | 8 |
| Context / Time | hour, month, is_night, is_peak_hour, lighting, is_poor_lighting | 6 |
| Engineered | aggressive_score, env_complexity, object_density | 3 |
| Basic Visual | brightness, contrast | 2 |
| YOLO Object Detection | object counts per category | 9 |
| CNN Embeddings (PCA) | cnn_feature_0 to cnn_feature_9 | 10 |
| ViT Embeddings (PCA) | vit_feature_0 to vit_feature_9 | 10 |
| **Total** | | **48** |

## Next Notebook
→ **Notebook 5** loads this dataset, creates proxy risk labels, trains models, and evaluates the framework

In [ ]:
# ============================================================
# STEP 0: CONNECT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

print("Drive connected.")

Mounted at /content/drive
Drive connected.


In [ ]:
# ============================================================
# STEP 1: LOAD DATASETS
# ============================================================
import pandas as pd

base_path = "/content/drive/My Drive/Z/Study/DBS/SEM 2/Dissertation/Google Colab"

ego_df = pd.read_csv(f"{base_path}/02_ego_features/ego_features_2000.csv")
visual_df = pd.read_csv(f"{base_path}/03_visual_features/visual_features_final.csv")
prepared_df = pd.read_csv(f"{base_path}/01_data_preparation/prepared_dataset_2000.csv")

print("Ego:", ego_df.shape)
print("Visual:", visual_df.shape)
print("Prepared:", prepared_df.shape)

Ego: (2000, 10)
Visual: (1899, 32)
Prepared: (2000, 16)


In [ ]:
# ============================================================
# STEP 2: MERGE DATASETS
# ============================================================
# PURPOSE:Merge ego-motion, visual, and metadata features on clip_id.
#         anomaly_score is deliberately NOT merged in — it will be
#         created fresh in Notebook 5 on training data only.
# OUTPUT:Unified feature dataset without target leakage.

# ------------------------------------------------------------
# DROP chunk FROM EGO TO AVOID COLUMN CLASH
# ------------------------------------------------------------
ego_df_clean = ego_df.drop(columns=["chunk"], errors="ignore")

# ------------------------------------------------------------
# MERGE EGO + VISUAL
# ------------------------------------------------------------
merged_df = ego_df_clean.merge(visual_df, on="clip_id", how="inner")

print("After ego + visual merge:", merged_df.shape)

# ------------------------------------------------------------
# MERGE METADATA (NO anomaly_score)
# ------------------------------------------------------------
meta_cols = [
    "clip_id", "chunk", "country", "month", "hour_of_day",
    "platform_class", "radar_config", "cluster", "scenario_type",
    "is_night", "is_peak_hour", "lighting_condition"
]

# Only keep meta_cols that actually exist in prepared_df
meta_cols_available = [c for c in meta_cols if c in prepared_df.columns]

merged_df = merged_df.merge(
    prepared_df[meta_cols_available],
    on="clip_id",
    how="inner"
)

print("After metadata merge:", merged_df.shape)
print("\nColumns:", list(merged_df.columns))
print("\nMissing values per column:")
print(merged_df.isna().sum()[merged_df.isna().sum() > 0])

print("\nFirst 5 rows:")
display(merged_df.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 2 complete: Datasets merged. anomaly_score excluded.")
print("------------------------------------------------------------")

After ego + visual merge: (989, 40)
After metadata merge: (989, 50)

Columns: ['clip_id', 'avg_speed', 'max_speed', 'avg_acceleration', 'max_acceleration', 'avg_jerk', 'max_jerk', 'braking_events', 'hard_braking_events', 'brightness', 'contrast', 'person_count', 'bicycle_count', 'car_count', 'motorcycle_count', 'bus_count', 'truck_count', 'traffic_light_count', 'stop_sign_count', 'total_objects', 'cnn_feature_0', 'cnn_feature_1', 'cnn_feature_2', 'cnn_feature_3', 'cnn_feature_4', 'cnn_feature_5', 'cnn_feature_6', 'cnn_feature_7', 'cnn_feature_8', 'cnn_feature_9', 'vit_feature_0', 'vit_feature_1', 'vit_feature_2', 'vit_feature_3', 'vit_feature_4', 'vit_feature_5', 'vit_feature_6', 'vit_feature_7', 'vit_feature_8', 'vit_feature_9', 'chunk', 'country', 'month', 'hour_of_day', 'platform_class', 'radar_config', 'cluster', 'scenario_type', 'is_night', 'is_peak_hour']

Missing values per column:
radar_config    284
dtype: int64

First 5 rows:



------------------------------------------------------------
Step 2 complete: Datasets merged. anomaly_score excluded.
------------------------------------------------------------


In [ ]:
# ============================================================
# STEP 3: FEATURE ENGINEERING
# ============================================================
# PURPOSE:Create composite features from existing columns.
#         aggressive_score uses StandardScaler to normalise
#         components before summing — prevents scale dominance.
# OUTPUT:Three engineered features added to merged_df.

from sklearn.preprocessing import StandardScaler
import numpy as np

# ------------------------------------------------------------
# AGGRESSIVE SCORE — SCALED COMPONENTS
# ------------------------------------------------------------
# Normalise each component before summing to prevent
# max_speed or max_jerk dominating due to scale differences

agg_components = merged_df[[
    "max_speed",
    "max_acceleration",
    "hard_braking_events",
    "max_jerk"
]].copy()

scaler_agg = StandardScaler()
agg_scaled = scaler_agg.fit_transform(agg_components)

merged_df["aggressive_score"] = agg_scaled.sum(axis=1)

# ------------------------------------------------------------
# ENVIRONMENTAL COMPLEXITY — WEIGHTED OBJECT COUNT
# ------------------------------------------------------------
merged_df["env_complexity"] = (
    merged_df["person_count"]        * 3.0 +
    merged_df["bicycle_count"]       * 2.0 +
    merged_df["motorcycle_count"]    * 2.0 +
    merged_df["car_count"]           * 1.0 +
    merged_df["truck_count"]         * 1.5 +
    merged_df["bus_count"]           * 1.5 +
    merged_df["traffic_light_count"] * 1.0
)

print("Feature engineering complete.")
print("\naggressive_score stats:")
print(merged_df["aggressive_score"].describe())

print("\nenv_complexity stats:")
print(merged_df["env_complexity"].describe())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 3 complete: Engineered features created with proper scaling.")
print("------------------------------------------------------------")

Feature engineering complete.

aggressive_score stats:
count    9.890000e+02
mean    -4.598052e-16
std      2.822428e+00
min     -5.102318e+00
25%     -1.886469e+00
50%     -4.970707e-01
75%      1.175929e+00
max      1.460927e+01
Name: aggressive_score, dtype: float64

env_complexity stats:
count    989.000000
mean       5.969666
std        6.958458
min        0.000000
25%        1.000000
50%        4.000000
75%        9.000000
max       54.500000
Name: env_complexity, dtype: float64

------------------------------------------------------------
Step 3 complete: Engineered features created with proper scaling.
------------------------------------------------------------


In [ ]:
# ============================================================
# STEP 4: DATA IMPROVEMENTS
# ============================================================
# PURPOSE:Add object density, clip extremes, and lighting label.
#         object_density uses total_objects per clip (not divided
#         by brightness — that formula was not defensible).
# OUTPUT:Refined dataset with additional quality features.

# ------------------------------------------------------------
# OBJECT DENSITY — TOTAL OBJECTS NORMALISED BY ENV COMPLEXITY
# ------------------------------------------------------------
# Simple, interpretable: how many objects relative to scene complexity
merged_df["object_density"] = (
    merged_df["total_objects"] /
    (merged_df["env_complexity"] + 1)  # +1 to avoid div by zero
)

# ------------------------------------------------------------
# CLIP JERK EXTREMES FOR STABILITY
# ------------------------------------------------------------
merged_df["max_jerk"] = merged_df["max_jerk"].clip(-50, 50)

# ------------------------------------------------------------
# LIGHTING CONDITION LABEL (CATEGORICAL — FOR ENCODING IN NB5)
# ------------------------------------------------------------
merged_df["lighting_condition"] = pd.cut(
    merged_df["brightness"],
    bins=[0, 40, 100, 255],
    labels=["poor", "moderate", "good"]
)

# Binary poor lighting flag
merged_df["is_poor_lighting"] = (merged_df["brightness"] < 40).astype(int)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------
print("Data improvements applied.\n")

print("object_density stats:")
print(merged_df["object_density"].describe())

print("\nlighting_condition distribution:")
print(merged_df["lighting_condition"].value_counts())

print("\nis_poor_lighting distribution:")
print(merged_df["is_poor_lighting"].value_counts())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 4 complete: Data improvements applied.")
print("------------------------------------------------------------")

Data improvements applied.

object_density stats:
count    989.000000
mean       0.577324
std        0.328686
min        0.000000
25%        0.461538
50%        0.666667
75%        0.833333
max        2.000000
Name: object_density, dtype: float64

lighting_condition distribution:
lighting_condition
moderate    786
poor        180
good         23
Name: count, dtype: int64

is_poor_lighting distribution:
is_poor_lighting
0    809
1    180
Name: count, dtype: int64

------------------------------------------------------------
Step 4 complete: Data improvements applied.
------------------------------------------------------------


In [ ]:
# ============================================================
# STEP 5: CLEAN DATASET
# ============================================================
# PURPOSE:Remove any rows with missing values in key feature columns.
#         anomaly_score is NOT present — no leakage check needed.
# OUTPUT:Clean final dataset ready for modelling.

# ------------------------------------------------------------
# CHECK MISSING VALUES
# ------------------------------------------------------------
missing = merged_df.isna().sum()
missing = missing[missing > 0]

print("Missing values before cleaning:")
print(missing if len(missing) > 0 else "None — dataset is clean.")

# ------------------------------------------------------------
# DROP ROWS WITH ANY MISSING FEATURE VALUES
# ------------------------------------------------------------
before = len(merged_df)
merged_df = merged_df.dropna()
after = len(merged_df)

print(f"\nRows dropped: {before - after}")
print(f"Final dataset shape: {merged_df.shape}")

# ------------------------------------------------------------
# VERIFY NO anomaly_score IN DATASET
# ------------------------------------------------------------
if "anomaly_score" in merged_df.columns:
    print("\n⚠️  WARNING: anomaly_score found in dataset — remove before saving.")
else:
    print("\n✅  Confirmed: anomaly_score NOT present. No leakage.")

# ------------------------------------------------------------
# FINAL COLUMN LIST
# ------------------------------------------------------------
print("\nFinal columns:")
print(list(merged_df.columns))

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 5 complete: Dataset cleaned. No target leakage present.")
print("------------------------------------------------------------")

Missing values before cleaning:
radar_config    284
dtype: int64

Rows dropped: 284
Final dataset shape: (705, 55)

✅  Confirmed: anomaly_score NOT present. No leakage.

Final columns:
['clip_id', 'avg_speed', 'max_speed', 'avg_acceleration', 'max_acceleration', 'avg_jerk', 'max_jerk', 'braking_events', 'hard_braking_events', 'brightness', 'contrast', 'person_count', 'bicycle_count', 'car_count', 'motorcycle_count', 'bus_count', 'truck_count', 'traffic_light_count', 'stop_sign_count', 'total_objects', 'cnn_feature_0', 'cnn_feature_1', 'cnn_feature_2', 'cnn_feature_3', 'cnn_feature_4', 'cnn_feature_5', 'cnn_feature_6', 'cnn_feature_7', 'cnn_feature_8', 'cnn_feature_9', 'vit_feature_0', 'vit_feature_1', 'vit_feature_2', 'vit_feature_3', 'vit_feature_4', 'vit_feature_5', 'vit_feature_6', 'vit_feature_7', 'vit_feature_8', 'vit_feature_9', 'chunk', 'country', 'month', 'hour_of_day', 'platform_class', 'radar_config', 'cluster', 'scenario_type', 'is_night', 'is_peak_hour', 'aggressive_score',

In [ ]:
# ============================================================
# STEP 6: SAVE DATASET
# ============================================================
import os

output_path = f"{base_path}/04_dataset"
os.makedirs(output_path, exist_ok=True)

csv_path = f"{output_path}/final_dataset.csv"
parquet_path = f"{output_path}/final_dataset.parquet"

merged_df.to_csv(csv_path, index=False)
merged_df.to_parquet(parquet_path, index=False)

print("Saved at:")
print(csv_path)
print(parquet_path)

Saved at:
/content/drive/My Drive/Z/Study/DBS/SEM 2/Dissertation/Google Colab/04_dataset/final_dataset.csv
/content/drive/My Drive/Z/Study/DBS/SEM 2/Dissertation/Google Colab/04_dataset/final_dataset.parquet
